# Goal
Read notebook for more details:

In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from pathlib import Path
import os
from dotenv import load_dotenv; load_dotenv()
import ipynbname
import shutil

# MUST BE FIRST - before any imports from cell_type_mapper --> error if FromSpecifiedMarkersRunner run wiht GPU
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import cell_type_mapper
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

# Load data

Your query data must be:
- **Format**: `.h5ad` file (AnnData format)
- **Structure**: 
  - `X` layer with **RAW** gene expression data (cells × genes)
  - `obs` with cell metadata
  - `var` with gene names, **Ensembl IDs** for MapMyCells-supported taxonomies

In [2]:
TISSUE = "SN"
QUERY_PATH = f"/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/{TISSUE}/{TISSUE}_combined_QC.h5ad"

# paths to files where mapping output will be written
json_dst_path = str(Path(QUERY_PATH).parent / "map_my_cell" / "mapping.json")
csv_dst_path = str(Path(QUERY_PATH).parent / "map_my_cell" / "mapping.csv")
os.makedirs(os.path.dirname(json_dst_path), exist_ok=True)
os.makedirs(os.path.dirname(csv_dst_path), exist_ok=True)

# saving adata path
adata_labelled_path = f"{os.path.splitext(QUERY_PATH)[0]}_mmc.h5ad"
adata_labelled_path

'/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/SN/SN_combined_QC_mmc.h5ad'

# Define Atlas/ref 
### SILLETTI CORTEX ATALS FOR BFC
### BG FOR SN AND STR

In [3]:
if TISSUE == "DFC":
    query_marker_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/WHB-10Xv3_20240831/query_markers.n10.20240221800.json"
    precomputed_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/WHB-10Xv3_20240831/precomputed_stats.siletti.training.h5"
elif (TISSUE == "Striatum") or (TISSUE == "SN"):
    query_marker_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.query_markers.20250507.json"
    precomputed_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.precomputed_stats.20250507.h5"

# Check gene overlapping

In [4]:
import json
import anndata as ad

# Load reference marker genes
with open(query_marker_path) as f:
    markers = json.load(f)

ref_genes = set()
for genes in markers.values():
    ref_genes.update(genes)

# Load your adata
adata = ad.read_h5ad(QUERY_PATH, backed="r")
query_genes = set(adata.var_names)

overlap = ref_genes & set(adata.var_names)
print(f"Reference genes: {len(ref_genes)}")
print(f"Query genes:     {len(adata.var_names)}")
print(f"Overlap:         {len(overlap)} ({len(overlap)/len(ref_genes)*100:.1f}%)")

del adata


Reference genes: 6852
Query genes:     38601
Overlap:         6514 (95.1%)


# Run Mapping

Now we will actually [perform the mapping](https://github.com/AllenInstitute/cell_type_mapper/blob/main/docs/mapping_cells.md).

In [5]:
config = {
    # output paths
    "query_path": QUERY_PATH,
    "extended_result_path": json_dst_path,
    "csv_result_path": csv_dst_path,
    "verbose_csv": True,

    # inout paths
    "query_markers": {
       "serialized_lookup": query_marker_path
    },
    "precomputed_stats": {
        "path": precomputed_path
    },

    "type_assignment": {
        "n_processors": 32,
        "normalization": "raw", # Use raw counts (not normalized)
        "bootstrap_factor": 0.5,
        "bootstrap_iteration": 100
    }
}

In [6]:
runner = FromSpecifiedMarkersRunner(
    args=[],
    input_data=config
)
runner.run()
print("Done!")

=== Running Hierarchical Mapping 1.5.2 with config ===
{
  "precomputed_stats": {
    "log_level": "ERROR",
    "path": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.precomputed_stats.20250507.h5"
  },
  "query_markers": {
    "log_level": "ERROR",
    "serialized_lookup": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.query_markers.20250507.json",
    "collapse_markers": false
  },
  "flatten": false,
  "extended_result_path": "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/SN/map_my_cell/mapping.json",
  "drop_level": null,
  "log_path": null,
  "query_gene_id_col": null,
  "query_path": "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/SN/SN_combined_QC.h5ad",
  "cloud_safe": false,
  "summary_metadata_path": null,
  "obsm_key": null,
  "verbose_stdout": true,
  "type_assignment": {
    "chunk_size": 10000,
    "normalization": "raw",
    "bootstrap_iteration": 100,
    "min_markers": 10,
    "n_processors": 32,
    "bootstrap

/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:73: UserWarning: numpy's internal parallelization is enabled. This could cause independent worker processes to compete for resources, degrading performance. We recommend setting the following environment variables to '1' to improve performance
{
  "NUMEXPR_NUM_THREADS": "",
  "MKL_NUM_THREADS": "",
  "OMP_NUM_THREADS": ""
}
  warnings.warn(msg)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:104: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  self.env(f"anndata version: {anndata.__version__}")
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/taxonomy/utils.py:253: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "
/home/gdallagl/myworkdir/XDP/.venv/lib

BENCHMARK: spent 1.4159e-01 seconds creating query marker cache
Running CPU implementation of type assignment.
BENCHMARK: spent 7.9077e+02 seconds assigning cell types
Writing marker genes to output file
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
CLEANING UP
Done!


# Output of mapping file

The results of our mapping are now in two files: the csv file pointed to by `csv_dst_path` and the JSON file pointed to by `json_dst_path`. Dedicated documentation of the the contents of the mapping output [can be found here.](https://github.com/AllenInstitute/cell_type_mapper/blob/main/docs/output.md)

## CSV output file

The CSV file is effectively just a dataframe. For every cell at every taxonomy level, you have its assigned cell type (both as a guaranteed unique "label" and a more human readable "name") along with quality metrics assessing the confidence in the mapping (see the detailed documentation above).

In [7]:
mapping_csv = pd.read_csv(csv_dst_path, comment='#')
mapping_csv = mapping_csv.set_index("cell_id")

display(mapping_csv)

,Neighborhood_label,Neighborhood_name,Neighborhood_bootstrapping_probability,Neighborhood_aggregate_probability,Neighborhood_correlation_coefficient,Class_label,Class_name,Class_bootstrapping_probability,Class_aggregate_probability,Class_correlation_coefficient,...,Group_name,Group_bootstrapping_probability,Group_aggregate_probability,Group_correlation_coefficient,Cluster_label,Cluster_name,Cluster_alias,Cluster_bootstrapping_probability,Cluster_aggregate_probability,Cluster_correlation_coefficient
cell_id,,,,,,,,,,,,,,,,,,,,,
22CTCMLT4__pXDPsHSrSNid240830rxn4__ACGGCCAGTCTTCAAG-1_SN_SCF-19-014,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7932,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8665,...,Astrocyte,1.0,1.0,0.8230,CS20250428_CLUST_0171,Human-472,Human-472,0.72,0.72,0.5723
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGCGGTAGTTAAAGTG-1_SN_SCF-20-025,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7428,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8161,...,Astrocyte,1.0,1.0,0.7593,CS20250428_CLUST_0165,Human-150,Human-150,0.44,0.44,0.5016
22CTCMLT4__pXDPsHSrSNid240830rxn4__TGCCCTAAGTCGTACT-1_SN_SCF-20-025,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7409,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8271,...,Astrocyte,1.0,1.0,0.7402,CS20250428_CLUST_0164,Human-149,Human-149,0.96,0.96,0.4695
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGTCACTGTGAGGCTA-1_SN_SCF-20-025,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7654,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8333,...,Astrocyte,1.0,1.0,0.7734,CS20250428_CLUST_0164,Human-149,Human-149,0.46,0.46,0.5376
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGACTTCGTGAGCGAT-1_SN_SCF_22-043,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7761,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8662,...,Astrocyte,1.0,1.0,0.8146,CS20250428_CLUST_0171,Human-472,Human-472,1.00,1.00,0.5999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__GCTATTGAGGTAGGGC-1_SN_SCF-19-020,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.6620,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.7951,...,Oligo OPALIN,1.0,1.0,0.6446,CS20250428_CLUST_0227,Human-1,Human-1,0.99,0.99,0.5215
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__CACAATCCACACACGC-1_SN_SCF-22-042,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.6165,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.7539,...,Oligo OPALIN,1.0,1.0,0.6179,CS20250428_CLUST_0227,Human-1,Human-1,0.71,0.71,0.3148
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__ATACGGACACAACGGG-1_SN_SCF_22-060,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.6379,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.7481,...,Oligo OPALIN,1.0,1.0,0.6009,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.00,0.4928


# Add Metadata to adata and Save

In [8]:
# Read adata
query_adata = sc.read_h5ad(QUERY_PATH)

# Merge metadata
query_adata.obs = query_adata.obs.join(mapping_csv, how="left")
display(query_adata.obs)

# save 
query_adata.write(adata_labelled_path)


n_cells_original = len(query_adata.obs)
n_cells_mapped = mapping_csv.index.nunique()
print(f"Original cells: {n_cells_original}")
print(f"Mapped cells: {n_cells_mapped}")
assert n_cells_original == n_cells_mapped, "Mapping incomplete!"

,background_fraction,cell_probability,cell_size,droplet_efficiency,barcode,bcl,rna_index,library,library__barcode,frac_mito,...,Group_name,Group_bootstrapping_probability,Group_aggregate_probability,Group_correlation_coefficient,Cluster_label,Cluster_name,Cluster_alias,Cluster_bootstrapping_probability,Cluster_aggregate_probability,Cluster_correlation_coefficient
barcode,,,,,,,,,,,,,,,,,,,,,
22CTCMLT4__pXDPsHSrSNid240830rxn4__ACGGCCAGTCTTCAAG-1_SN_SCF-19-014,0.001881,0.999955,13558.819336,2.304307,22CTCMLT4__pXDPsHSrSNid240830rxn4__ACGGCCAGTCT...,22CTCMLT4,pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4__ACGGCCAGTCT...,0.000103,...,Astrocyte,1.0,1.0,0.8230,CS20250428_CLUST_0171,Human-472,Human-472,0.72,0.72,0.5723
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGCGGTAGTTAAAGTG-1_SN_SCF-20-025,0.005146,0.999955,12225.272461,2.241633,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGCGGTAGTTA...,22CTCMLT4,pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGCGGTAGTTA...,0.000079,...,Astrocyte,1.0,1.0,0.7593,CS20250428_CLUST_0165,Human-150,Human-150,0.44,0.44,0.5016
22CTCMLT4__pXDPsHSrSNid240830rxn4__TGCCCTAAGTCGTACT-1_SN_SCF-20-025,0.003087,0.999955,12147.070312,2.183491,22CTCMLT4__pXDPsHSrSNid240830rxn4__TGCCCTAAGTC...,22CTCMLT4,pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4__TGCCCTAAGTC...,0.000041,...,Astrocyte,1.0,1.0,0.7402,CS20250428_CLUST_0164,Human-149,Human-149,0.96,0.96,0.4695
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGTCACTGTGAGGCTA-1_SN_SCF-20-025,0.006107,0.999955,10735.762695,2.109610,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGTCACTGTGA...,22CTCMLT4,pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGTCACTGTGA...,0.000242,...,Astrocyte,1.0,1.0,0.7734,CS20250428_CLUST_0164,Human-149,Human-149,0.46,0.46,0.5376
22CTCMLT4__pXDPsHSrSNid240830rxn4__CGACTTCGTGAGCGAT-1_SN_SCF_22-043,0.004922,0.999955,10420.652344,2.043174,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGACTTCGTGA...,22CTCMLT4,pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4,22CTCMLT4__pXDPsHSrSNid240830rxn4__CGACTTCGTGA...,0.000000,...,Astrocyte,1.0,1.0,0.8146,CS20250428_CLUST_0171,Human-472,Human-472,1.00,1.00,0.5999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__GCTATTGAGGTAGGGC-1_SN_SCF-19-020,0.473545,0.999668,7679.466309,0.512645,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__GCTAT...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__GCTAT...,0.000000,...,Oligo OPALIN,1.0,1.0,0.6446,CS20250428_CLUST_0227,Human-1,Human-1,0.99,0.99,0.5215
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__CACAATCCACACACGC-1_SN_SCF-22-042,0.510791,0.999870,7597.659668,0.520223,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__CACAA...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__CACAA...,0.000000,...,Oligo OPALIN,1.0,1.0,0.6179,CS20250428_CLUST_0227,Human-1,Human-1,0.71,0.71,0.3148
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__ATACGGACACAACGGG-1_SN_SCF_22-060,0.527286,0.999277,7346.480469,0.540129,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__ATACG...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G4__ATACG...,0.000000,...,Oligo OPALIN,1.0,1.0,0.6009,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.00,0.4928


Original cells: 125635
Mapped cells: 125635


In [9]:
# print(query_adata.obs.columns)#Group_names.value_counts()
# query_adata.obs.supercluster_name.value_counts()